# FitzHugh–Nagumo PINN bonus

**Mathematical Modelling Group Project (MAC F314)**

## Can a few voltage readings reveal a hidden model parameter?

The [core notebook](fhn_project.ipynb) takes the model parameters as inputs and predicts a neuron's response. This notebook turns that problem around. We generate voltage readings ourselves, hide one parameter $a$ from the fitting algorithms, and see how accurately each method can recover it.

| Stage | What you will do |
| --- | --- |
| **Set up the experiment** | Simulate the model and construct clean and sparse, noisy voltage readings. |
| **Fit the hidden parameter** | Train a PyTorch physics-informed neural network (PINN) and fit a SciPy least-squares baseline using identical observations. |
| **Check the result** | Re-simulate the model with each fitted parameter at a different current. Reserve one case for the final test. |

Open this notebook in **Google Colab** and run the cells from top to bottom. The first run uses a Colab CPU by default. The notebook prints progress while it trains. Step 10 leaves the final test switched off until you are ready to evaluate it. All data are generated in the notebook. [Google Colab FAQ](https://research.google.com/colaboratory/faq.html)

**Scientific libraries.** SciPy solves the ODE and performs least-squares fitting. PyTorch trains the PINN. Matplotlib draws the graphs. Python's built-in `random` module supplies the reproducible measurement noise.

## 🟦 1. Set up the Colab experiment

We estimate one parameter at a time. The recovery feedback $b$, recovery rate $\varepsilon$, current $I$, and starting state are supplied to both fitting methods. Both methods start with the same guess for $a$ and use the same allowed interval. The interval is a modelling choice for this synthetic experiment.

A random seed makes the synthetic observations and network initialization repeatable on the same setup. Record the printed package versions and device when reporting your results.

<pre style="background-color: #E7F1FF; color: #17365D; padding: 12px; border-left: 4px solid #2563EB;"><code>SETUP
Choose the known physical parameters and the interval for the hidden one.
Set the observation seed, training seed, and optimizer budget.
Select the Colab compute device and print the environment details.</code></pre>

In [ ]:
import random

import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
import torch
from torch import nn

# Use double precision for the network, the state, and the derivatives.
torch.set_default_dtype(torch.float64)
USE_GPU = False
DEVICE = torch.device("cuda" if USE_GPU and torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    # A small dense network usually needs little CPU thread parallelism.
    torch.set_num_threads(1)

B = 0.80
EPSILON = 0.08
A_START = 0.50
A_MIN, A_MAX = 0.20, 1.20
END_TIME = 40.0
TIME_STEP = 0.10
NOISE_SEED = 2026
FIT_SEED = 7
NOISE_STD = 0.03

# These settings are shared by the clean and noisy PINN runs.
WARMUP_STEPS = 400
ADAM_STEPS = 1800
LBFGS_STEPS = 300
COLLOCATION_COUNT = 241

print(f"PyTorch {torch.__version__} | SciPy {scipy.__version__} | Matplotlib {matplotlib.__version__}")
print(f"Device: {DEVICE} | dtype: {torch.get_default_dtype()} | seeds: data={NOISE_SEED}, fit={FIT_SEED}")

## 🟩 2. Define the equations and the three cases

We use the same model as the core notebook.

$$
\frac{dv}{dt}=v-\frac{v^3}{3}-w+I,
\qquad
\frac{dw}{dt}=\varepsilon(v+a-bw).
$$

The **forward problem** takes $a$ and calculates the voltage and recovery curves. The **inverse problem** takes voltage readings and estimates $a$. Recovery $w$ is available in our simulated reference, but we treat it as **unobserved** during fitting.

| Case | Current | Starting state $(v_0,w_0)$ | Purpose |
| --- | --- | --- | --- |
| Training | $0.50$ | $(-1.15,-0.62)$ | Fit the hidden parameter from voltage readings. |
| Validation | $0.65$ | $(-1.00,-0.55)$ | See whether the estimate works at a different current and starting state. |
| Final test | $0.90$ | $(-1.20,-0.70)$ | Evaluate once after freezing the fitting settings. |

Each run covers 40 model time units. The three cases share the same true model parameters.

<pre style="background-color: #E8F7ED; color: #15432B; padding: 12px; border-left: 4px solid #27804E;"><code>FORWARD MODEL
Evaluate the two FitzHugh–Nagumo slopes.
Give a current, starting state, and proposed a to SciPy.
Return the voltage and recovery on the requested time grid.</code></pre>

In [ ]:
CASES = {
    "train": (0.50, (-1.15, -0.62)),
    "validation": (0.65, (-1.00, -0.55)),
    "test": (0.90, (-1.20, -0.70)),
}
sample_count = round(END_TIME / TIME_STEP)
reference_times = [index * TIME_STEP for index in range(sample_count + 1)]

def fhn_rhs(state, current, a_value):
    # The same equations work with ordinary numbers and PyTorch tensors.
    voltage, recovery = state
    dv_dt = voltage - voltage**3 / 3 - recovery + current
    dw_dt = EPSILON * (voltage + a_value - B * recovery)
    return dv_dt, dw_dt

def simulate(case, a_value, times, *, rtol=1e-9, atol=1e-11):
    current, initial_state = case
    # Integrate from the known initial state, even if the first reading is later.
    solution = solve_ivp(
        lambda _time, state: fhn_rhs(state, current, a_value),
        (0.0, times[-1]),
        initial_state,
        t_eval=times,
        method="DOP853",
        rtol=rtol,
        atol=atol,
    )
    if not solution.success:
        raise RuntimeError(solution.message)
    return solution.y[0], solution.y[1]

# Spot-check the two slopes against the hand calculation in the core notebook.
dv_dt, dw_dt = fhn_rhs((0.0, 0.0), 0.50, 0.70)
assert abs(dv_dt - 0.50) < 1e-12 and abs(dw_dt - 0.056) < 1e-12

## 🟨 3. Generate the training observations

We choose $a_{\rm true}=0.70$ and solve the ODE with tighter tolerances to get a clean numerical reference. We then build two datasets from its voltage component.

- **Clean.** Keep every fourth sample and add no noise.
- **Sparse and noisy.** Divide the positive-time samples into blocks of ten. Pick one sample per block and add an independent Gaussian error with standard deviation $0.03$.

The fitting functions receive only the selected times and voltages. The dense curve, true $a$, and reference recovery values are for generating data and evaluating the results.

<pre style="background-color: #FFF3D4; color: #503300; padding: 12px; border-left: 4px solid #B56A00;"><code>DATA
Simulate one clean training trajectory with the true parameter.
Sample the voltage to make a clean dataset and a noisier, sparser one.
Plot the observations so their coverage is easy to check.</code></pre>

In [ ]:
TRUE_A = 0.70

def reference_trace(case):
    # Tighten the solver tolerances for the synthetic ground truth.
    return simulate(case, TRUE_A, reference_times, rtol=1e-11, atol=1e-13)

train_voltage, train_recovery = reference_trace(CASES["train"])
rng = random.Random(NOISE_SEED)
clean_indices = range(0, len(reference_times), 4)

# Stratify the noisy samples so every time block contributes a reading.
noisy_indices = []
for start in range(1, len(reference_times), 10):
    stop = min(start + 10, len(reference_times))
    noisy_indices.append(rng.randrange(start, stop))

datasets = {
    "Clean": (
        [reference_times[index] for index in clean_indices],
        [float(train_voltage[index]) for index in clean_indices],
    ),
    "Sparse and noisy": (
        [reference_times[index] for index in noisy_indices],
        [float(train_voltage[index]) + rng.gauss(0.0, NOISE_STD) for index in noisy_indices],
    ),
}

# Sanity-check the sampling design before either method sees the data.
assert len(datasets["Clean"][0]) == 101
assert len(datasets["Sparse and noisy"][0]) == 40
assert noisy_indices == sorted(set(noisy_indices))

data_figure, axes = plt.subplots(1, 2, figsize=(10.2, 3.5), sharey=True, layout="constrained")
for ax, (label, (times, observations)) in zip(axes, datasets.items()):
    ax.plot(reference_times, train_voltage, color="0.3", label="Clean reference")
    ax.scatter(times, observations, s=14, color="tab:orange", label="Observed voltage", zorder=3)
    ax.set(title=f"{label} ({len(times)} readings)", xlabel="Time (model units)")
axes[0].set_ylabel("Voltage v")
axes[0].legend(framealpha=1, fontsize=8)
plt.show()

## 🟪 4. Build the time-to-state network

The network takes one time value and returns two numbers, $\hat v(t)$ and $\hat w(t)$. A hat denotes a network prediction. We use smooth `tanh` activations because the training loss will need the derivatives of those curves.

The input includes normalized time and a few fixed sine and cosine features. They give the small network smooth ways to represent a sharp spike. We enforce the known initial state exactly with
$$
(\hat v,\hat w)(t)=(v_0,w_0)+(1-e^{-t})N_\theta(t).
$$
At $t=0$ the multiplier is zero. As time increases the network can change both states. We use a sigmoid transform to keep the fitted $a$ within the same interval used by least squares.

<pre style="background-color: #F1EAFE; color: #38205C; padding: 12px; border-left: 4px solid #7547B2;"><code>NETWORK
Scale time and build smooth time features.
Predict two state corrections with a small dense network.
Apply the exact starting state and map a into its allowed interval.</code></pre>

In [ ]:
def time_column(values):
    # Each row is one time point. The network expects a column-shaped tensor.
    return torch.as_tensor(values, device=DEVICE).reshape(-1, 1)

class FitzHughPINN(nn.Module):
    def __init__(self, initial_state):
        super().__init__()
        self.register_buffer("initial_state", torch.tensor(initial_state).reshape(1, 2))

        # Invert the sigmoid so the constrained parameter starts at A_START.
        fraction = (A_START - A_MIN) / (A_MAX - A_MIN)
        self.raw_a = nn.Parameter(torch.tensor(fraction).logit())

        # One time column plus three sine and three cosine columns.
        self.register_buffer("frequencies", torch.tensor([1.0, 2.0, 4.0]).reshape(1, 3))
        self.network = nn.Sequential(
            nn.Linear(7, 32), nn.Tanh(),
            nn.Linear(32, 32), nn.Tanh(),
            nn.Linear(32, 32), nn.Tanh(),
            nn.Linear(32, 2),
        )

    @property
    def a_value(self):
        return A_MIN + (A_MAX - A_MIN) * torch.sigmoid(self.raw_a)

    def forward(self, times):
        fraction = times / END_TIME
        angles = 2 * torch.pi * fraction * self.frequencies
        features = torch.cat((2 * fraction - 1, torch.sin(angles), torch.cos(angles)), dim=1)
        # The multiplier is exactly zero at time zero, fixing both initial states.
        return self.initial_state + (1 - torch.exp(-times)) * self.network(features)

# Verify the initial state and parameter transform before training.
torch.manual_seed(FIT_SEED)
probe = FitzHughPINN(CASES["train"][1]).to(DEVICE)
torch.testing.assert_close(probe(time_column([0.0])), probe.initial_state)
assert abs(probe.a_value.item() - A_START) < 1e-12

## 🟧 5. Turn the ODE into a training loss

There are two places where we evaluate the network.

| Where | What the loss checks |
| --- | --- |
| **Observation times** | Does predicted voltage agree with the measured voltage? |
| **Physics times** | Do both predicted slopes agree with the two ODEs? These times do not need voltage measurements. |

For each state, a **physics residual** is the network's time derivative minus the derivative given by the model. Substitute the predicted states and the current estimate $\hat a$ into the equations.

$$
\begin{aligned}
r_v &= \frac{d\hat v}{dt} - \left(\hat v-\frac{\hat v^3}{3}-\hat w+I\right),\\
r_w &= \frac{d\hat w}{dt} - \varepsilon(\hat v+\hat a-b\hat w).
\end{aligned}
$$

A good fit makes both residuals small and agrees with the observed voltage. The recovery residual has weight 4 to give it more influence alongside the faster voltage dynamics. The exact initial-state formula in Step 4 supplies the initial condition.

$$
\mathcal L=\operatorname{mean}\!\left[(\hat v-v_{\rm obs})^2\right]
 + \operatorname{mean}(r_v^2)+4\operatorname{mean}(r_w^2).
$$

Automatic differentiation follows the whole network from model time $t$ to the outputs, including the time scaling. [PyTorch automatic differentiation](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad.html)

<pre style="background-color: #FFEDE3; color: #522915; padding: 12px; border-left: 4px solid #B55223;"><code>LOSS
Compare predicted voltage to observations at their measurement times.
Differentiate both network outputs at the physics times.
Subtract the ODE slopes and combine the three mean-square terms.</code></pre>

In [ ]:
def voltage_data_loss(model, observed_times, observed_voltage):
    voltage_prediction = model(observed_times)[:, 0:1]
    return (voltage_prediction - observed_voltage).square().mean()

def physics_losses(model, physics_times, current):
    # Differentiate with respect to model time, before the network scales it.
    times = physics_times.detach().requires_grad_(True)
    states = model(times)
    voltage, recovery = states[:, 0:1], states[:, 1:2]

    # Each row depends on its own time. Differentiating the sum gives one slope per row.
    dv_dt = torch.autograd.grad(voltage.sum(), times, create_graph=True)[0]
    dw_dt = torch.autograd.grad(recovery.sum(), times, create_graph=True)[0]
    ode_v, ode_w = fhn_rhs((voltage, recovery), current, model.a_value)
    return (dv_dt - ode_v).square().mean(), (dw_dt - ode_w).square().mean()

def loss_terms(model, observed_times, observed_voltage, physics_times, current):
    data = voltage_data_loss(model, observed_times, observed_voltage)
    voltage, recovery = physics_losses(model, physics_times, current)
    return data + voltage + 4 * recovery, data, voltage, recovery

# Verify that the physics loss supplies a gradient to the hidden parameter.
probe_losses = loss_terms(
    probe, time_column([1.0]), time_column([0.0]),
    time_column([0.0, 2.0, 10.0]), CASES["train"][0],
)
probe_losses[0].backward()
assert probe.raw_a.grad is not None
assert torch.isfinite(probe.raw_a.grad).item() and probe.raw_a.grad.abs().item() > 0
del probe, probe_losses

## 🟫 6. Train the PINN in three stages

1. **Warm up on voltage.** Let the network start learning the observed curve. The parameter $a$ only appears in the physics loss, so it stays at its starting guess in this stage.
2. **Train jointly with Adam.** Fit the observations and the equations together. The network weights and the hidden parameter can now both change.
3. **Refine with L-BFGS.** Use the same joint objective and the same fixed physics grid. L-BFGS re-evaluates the loss and gradients through a function called a closure.

For comparison we start the clean and noisy runs with the same network seed and settings. An optimizer can converge to a poor estimate, so the parameter error and the held-out prediction matter more than a small loss alone.

<pre style="background-color: #F6EBDB; color: #523719; padding: 12px; border-left: 4px solid #9C641F;"><code>TRAIN
Fit the voltage observations briefly to initialize the curve.
Use Adam on the combined data and physics objective.
Use L-BFGS on the same objective and record the fitted a.</code></pre>

In [ ]:
def fit_pinn(case, times, observations, label):
    current, initial_state = case
    torch.manual_seed(FIT_SEED)
    model = FitzHughPINN(initial_state).to(DEVICE)
    observed_times = time_column(times)
    observed_voltage = time_column(observations)
    physics_times = torch.linspace(0.0, END_TIME, COLLOCATION_COUNT, device=DEVICE).reshape(-1, 1)

    adam = torch.optim.Adam(model.parameters(), lr=0.003)
    for _ in range(WARMUP_STEPS):
        # Warm up the curve before using the equations to estimate a.
        adam.zero_grad(set_to_none=True)
        voltage_data_loss(model, observed_times, observed_voltage).backward()
        adam.step()

    history = []
    log_interval = max(1, ADAM_STEPS // 10)
    for step in range(1, ADAM_STEPS + 1):
        adam.zero_grad(set_to_none=True)
        terms = loss_terms(model, observed_times, observed_voltage, physics_times, current)
        if step == 1 or step % log_interval == 0 or step == ADAM_STEPS:
            if not torch.isfinite(terms[0]).item():
                raise RuntimeError(f"Non-finite loss in the {label} fit at Adam step {step}")
            # The recorded loss is the one used for this optimizer step.
            history.append((step, *(term.detach().item() for term in terms)))
            if step == 1 or step % (5 * log_interval) == 0 or step == ADAM_STEPS:
                print(f"{label}: Adam {step:4d}/{ADAM_STEPS} | loss={history[-1][1]:.3e} | a={model.a_value.item():.5f}")
        terms[0].backward()
        adam.step()

    optimizer = torch.optim.LBFGS(
        model.parameters(), lr=1.0, max_iter=LBFGS_STEPS,
        history_size=25, line_search_fn="strong_wolfe",
    )

    def closure():
        # L-BFGS may request several fresh evaluations of the same objective.
        optimizer.zero_grad(set_to_none=True)
        total, _, _, _ = loss_terms(model, observed_times, observed_voltage, physics_times, current)
        total.backward()
        return total

    optimizer.step(closure)
    final_terms = tuple(
        term.detach().item()
        for term in loss_terms(model, observed_times, observed_voltage, physics_times, current)
    )
    if not all(-float("inf") < term < float("inf") for term in final_terms):
        raise RuntimeError(f"Non-finite final loss in the {label} fit")
    print(f"{label}: after L-BFGS | loss={final_terms[0]:.3e} | a={model.a_value.item():.5f}")
    return model, history, final_terms

# Each fit sees only the voltage readings for its own training dataset.
pinn_results = {
    label: fit_pinn(CASES["train"], times, observations, label)
    for label, (times, observations) in datasets.items()
}

## 🟦 7. Fit the SciPy least-squares baseline

This method tries a value of $a$, solves the FHN equations, and computes the voltage errors at the observed times. SciPy adjusts $a$ to reduce the sum of those squared errors. This is a strong baseline because the equations are known and only one parameter is being fitted.

Both methods get the same voltage readings, initial state, current, starting guess for $a$, and allowed interval. The **absolute parameter error** is $|\hat a-a_{\rm true}|$. Dividing this by $|a_{\rm true}|$ and multiplying by 100 gives the percentage error. The clean least-squares fit is also a check that the synthetic inverse problem is set up correctly. [SciPy least-squares documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html)

<pre style="background-color: #DCF6FA; color: #16424C; padding: 12px; border-left: 4px solid #16869A;"><code>BASELINE
Simulate the voltage at the observation times for a trial value of a.
Return the voltage errors to the least-squares optimizer.
Compare both fitted values against the known synthetic truth.</code></pre>

In [ ]:
def fit_least_squares(case, times, observations):
    def voltage_errors(candidate):
        # SciPy supplies a one-element array because only a is being estimated.
        predicted, _ = simulate(case, float(candidate[0]), times)
        return predicted - observations

    fit = least_squares(voltage_errors, [A_START], bounds=([A_MIN], [A_MAX]))
    if not fit.success:
        raise RuntimeError(fit.message)
    return float(fit.x[0])

estimates = {}
for label, (times, observations) in datasets.items():
    model, _, _ = pinn_results[label]
    estimates[label] = {
        "PINN": model.a_value.item(),
        "Least squares": fit_least_squares(CASES["train"], times, observations),
    }

# Clean synthetic observations should recover the true parameter with the ODE baseline.
assert abs(estimates["Clean"]["Least squares"] - TRUE_A) < 1e-5
print("Clean least-squares recovery passed.")
print(f"True a = {TRUE_A:.3f} | Starting guess = {A_START:.3f}")
print(f"{'Training data':<18} {'Method':<14} {'Estimated a':>12} {'Abs. error':>12} {'Rel. error':>12}")
for label, methods in estimates.items():
    for method, estimate in methods.items():
        error = abs(estimate - TRUE_A)
        relative = 100 * error / abs(TRUE_A)
        print(f"{label:<18} {method:<14} {estimate:12.6f} {error:12.6f} {relative:11.2f}%")

## 🟪 8. Inspect how each method fits the training data

First look at the three loss terms during joint Adam training. The warm-up optimizes voltage alone. The printed summary gives the three errors after L-BFGS.

Then compare the observed voltage with two fitted curves: the PINN's direct output and the ODE solved with the least-squares estimate. A good-looking PINN curve can still give a poor parameter estimate. The comparison in Step 9 tests what happens when each fitted parameter is used to solve a different case.

<pre style="background-color: #F5E8F3; color: #49213E; padding: 12px; border-left: 4px solid #994283;"><code>TRAINING DIAGNOSTICS
Plot the joint training loss terms over the Adam steps.
Compare both fitted training curves with the observations and clean reference.
Read the final physics losses separately from the voltage error.</code></pre>

In [ ]:
loss_figure, axes = plt.subplots(1, 2, figsize=(10.2, 3.5), sharey=True, layout="constrained")
for ax, (label, (_, history, final_terms)) in zip(axes, pinn_results.items()):
    steps, total, data_loss, voltage_loss, recovery_loss = zip(*history)

    # Clip only for the log plot so an exact zero is still displayable.
    curves = (
        (total, "Combined loss", "0.2"),
        (data_loss, "Voltage readings", "tab:blue"),
        (voltage_loss, "Voltage equation", "tab:orange"),
        ([4 * value for value in recovery_loss], "Recovery equation ×4", "tab:green"),
    )
    for values, name, color in curves:
        ax.semilogy(steps, [max(value, 1e-16) for value in values], label=name, color=color)
    ax.set(title=label, xlabel="Joint Adam step")
    print(f"{label} final losses: data={final_terms[1]:.2e}, "
          f"voltage physics={final_terms[2]:.2e}, recovery physics={final_terms[3]:.2e}")

axes[0].set_ylabel("Mean-square loss")
axes[0].legend(framealpha=1, fontsize=8)
plt.show()

The next plot compares the direct network curve and the least-squares ODE with the original readings. Focus on whether each curve captures both the fast spike and the slower recovery in voltage.

In [ ]:
training_figure, axes = plt.subplots(1, 2, figsize=(10.2, 3.5), sharey=True, layout="constrained")
for ax, (label, (times, observations)) in zip(axes, datasets.items()):
    model, _, _ = pinn_results[label]
    with torch.no_grad():
        # The direct network curve is specific to the training current and state.
        network_voltage = model(time_column(reference_times))[:, 0].cpu().tolist()
    baseline_voltage, _ = simulate(CASES["train"], estimates[label]["Least squares"], reference_times)

    ax.plot(reference_times, train_voltage, color="0.2", linewidth=2, label="Clean reference")
    ax.plot(reference_times, network_voltage, color="tab:purple", linestyle="--", label="PINN curve")
    ax.plot(reference_times, baseline_voltage, color="tab:green", linestyle=":", label="Least-squares ODE")
    ax.scatter(times, observations, color="tab:orange", s=11, label="Observed voltage", zorder=4)
    ax.set(title=label, xlabel="Time (model units)")

axes[0].set_ylabel("Training voltage v")
axes[0].legend(framealpha=1, fontsize=8)
plt.show()

## 🟩 9. Validate on an unseen trajectory

For validation we change the current and initial state. **Both methods now use the same ODE solver**. The only difference is which fitted value of $a$ goes into the solver. This is how we test parameter transfer. The time-to-state network from training only represents the original training case.

We report voltage and recovery **RMSE**, the square root of the mean squared prediction error. The reference recovery curve is available here because we simulated the data ourselves. We also count upward voltage crossings of zero. The spike timing error is the mean absolute difference between corresponding crossing times when both curves have the same nonzero spike count.

<pre style="background-color: #E1F5E8; color: #1B452D; padding: 12px; border-left: 4px solid #2C7C50;"><code>VALIDATION
Generate a clean reference at a new current and starting state.
Re-simulate the ODE with the PINN and least-squares estimates.
Compare both state curves and the timing of voltage spikes.</code></pre>

In [ ]:
def rmse(predicted, reference):
    # Both traces use the same output grid, so each sample has a direct counterpart.
    assert len(predicted) == len(reference) and len(reference) > 0
    mean_squared = sum((float(left) - float(right))**2 for left, right in zip(predicted, reference))
    return (mean_squared / len(reference))**0.5

def spike_times(times, voltage):
    crossings = []
    for left_t, right_t, left_v, right_v in zip(times, times[1:], voltage, voltage[1:]):
        if left_v < 0 <= right_v:
            # Estimate the crossing between the two adjacent voltage samples.
            fraction = -left_v / (right_v - left_v)
            crossings.append(left_t + fraction * (right_t - left_t))
    return crossings

def evaluate_unseen_case(case, case_label):
    reference_voltage, reference_recovery = reference_trace(case)
    reference_spikes = spike_times(reference_times, reference_voltage)
    figure, axes = plt.subplots(1, 2, figsize=(10.2, 3.5), sharey=True, layout="constrained")
    figure.suptitle(f"{case_label} (I = {case[0]:.2f})")
    print(f"{case_label} | I={case[0]:.2f} | reference spikes: {len(reference_spikes)}")
    print(f"{'Training data':<18} {'Method':<14} {'Voltage RMSE':>13} "
          f"{'Recovery RMSE':>14} {'Spikes':>7} {'Timing MAE':>12}")

    for ax, (label, methods) in zip(axes, estimates.items()):
        ax.plot(reference_times, reference_voltage, color="0.2", linewidth=2, label="Clean reference")
        for method, line_style, color in (
            ("PINN", "--", "tab:purple"),
            ("Least squares", ":", "tab:green"),
        ):
            voltage, recovery = simulate(case, methods[method], reference_times)
            crossings = spike_times(reference_times, voltage)
            timing_label = "n/a"
            if crossings and len(crossings) == len(reference_spikes):
                errors = [abs(left - right) for left, right in zip(crossings, reference_spikes)]
                timing_label = f"{sum(errors) / len(errors):.4f}"
            print(f"{label:<18} {method:<14} {rmse(voltage, reference_voltage):13.6f} "
                  f"{rmse(recovery, reference_recovery):14.6f} {len(crossings):7d} {timing_label:>12}")
            ax.plot(reference_times, voltage, color=color, linestyle=line_style, label=f"ODE with {method} a")

        ax.set(title=f"Fitted from {label.lower()} data", xlabel="Time (model units)")

    axes[0].set_ylabel("Voltage v")
    axes[0].legend(framealpha=1, fontsize=8)
    plt.show()
    return figure

# Check two simple metric cases before reporting experimental results.
assert rmse([1.0, 3.0], [0.0, 2.0]) == 1.0
assert spike_times([0.0, 2.0], [-1.0, 1.0]) == [1.0]
validation_figure = evaluate_unseen_case(CASES["validation"], "Validation case")

## 🟨 10. Evaluate the final test case

Review the validation results first. Once you have frozen the model, loss settings, optimizer budget, and observation design, change `RUN_FINAL_TEST` below to `True` and run this cell. Use this test to report final performance. Any changes based on these results would require a new held-out case for an independent evaluation.

<pre style="background-color: #F8F1DC; color: #4F3811; padding: 12px; border-left: 4px solid #9A6D17;"><code>FINAL TEST
Freeze the settings using the training and validation results.
Evaluate both fitted parameters at the reserved current and starting state.
Record voltage and recovery errors, spike counts, and timing errors.</code></pre>

In [ ]:
RUN_FINAL_TEST = False
test_figure = None
if RUN_FINAL_TEST:
    # Reuse the validation procedure without fitting on this case.
    test_figure = evaluate_unseen_case(CASES["test"], "Final test case")
else:
    print("Final test reserved. Set RUN_FINAL_TEST = True after freezing the settings.")

## ⬛ 11. Save figures for the report

The executed notebook contains the printed parameter estimates and error tables. If you need vector plots for the LaTeX report, set `EXPORT_FIGURES` to `True` and run this cell. Download the exported files and the executed notebook before closing the Colab session.

<pre style="background-color: #E9EDF4; color: #27344A; padding: 12px; border-left: 4px solid #4A607E;"><code>EXPORT
Save each available plot as a PDF for Overleaf and as an EPS for submission.
Include the final test plot only after the test has been run.</code></pre>

In [ ]:
EXPORT_FIGURES = False
if EXPORT_FIGURES:
    # pathlib is part of Python and is needed only to create the export folder.
    from pathlib import Path

    figure_dir = Path("pinn_figures")
    figure_dir.mkdir(exist_ok=True)
    figures = {
        "training_observations": data_figure,
        "joint_training_losses": loss_figure,
        "training_voltage_fits": training_figure,
        "validation_voltage": validation_figure,
    }
    if test_figure is not None:
        figures["final_test_voltage"] = test_figure
    for name, figure in figures.items():
        figure.savefig(figure_dir / f"{name}.pdf")
        figure.savefig(figure_dir / f"{name}.eps")
    print(f"Report figures saved as PDF and EPS in {figure_dir}.")

## How to interpret the results

- **Start with parameter error.** Compare both estimates with the known synthetic value. Then check whether the smaller parameter error also gives the better voltage and spike predictions on unseen cases.
- **Separate curve fitting from parameter recovery.** The network can match measured voltage while leaving a noticeable ODE residual or producing a biased estimate of $a$. Use the physics losses and the held-out ODE simulations to tell these apart.
- **Describe what the comparison measures.** The noisy dataset has fewer observations as well as added noise. The difference between the two results shows their combined effect. A separate sparse, noise-free run would help isolate the effect of fewer readings.
- **Report the limitations.** Both methods assume the exact FHN equations, known initial states, and known $b$ and $\varepsilon$. The synthetic benchmark checks recovery within those assumptions. A single noise draw and network seed do not measure how much results vary across runs. To assess variability, repeat the experiment with several recorded seeds and summarize the spread.

## References and acknowledgment

- FitzHugh, R. (1961). [Impulses and Physiological States in Theoretical Models of Nerve Membrane](https://pmc.ncbi.nlm.nih.gov/articles/PMC1366333/).
- Raissi, M., Perdikaris, P., and Karniadakis, G. E. (2019). [Physics-informed neural networks](https://doi.org/10.1016/j.jcp.2018.10.045).
- [PyTorch automatic differentiation](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad.html) and [SciPy least-squares documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html).

**AI acknowledgment.** ChatGPT assisted with planning the bonus experiment. OpenAI Codex assisted with writing and organizing this notebook, documenting the code, and checking the execution workflow.